In [1]:
# Imports
from pathlib import Path
import flammkuchen as fl
import torch
import numpy as np
import matplotlib.pyplot as plt
%matplotlib qt

# Metrics
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, precision_recall_curve, confusion_matrix, auc)

# Custom imports
import calcium_event_classifier as cec
import calcium_event_classifier.core.plot as cec_plot
from calcium_event_classifier.core.dffdataset import DffDataset
from calcium_event_classifier.core.classifier_dff import CalciumEventClassifierDff

# Set device (GPU or CPU)
device = cec.set_device()

AttributeError: module 'calcium_event_classifier' has no attribute 'set_device'

# Calcium Event Classifier (dFF) - Test & Evaluation

This notebook evaluates the trained ```CalciumEventClassifierDff``` model on a held-out test dataset. 
It:
- loads the model checkpoint
- runs inference on test data
- generates comprehensive performance metrics and visualizations

## 1. Load Model and Checkpoint

Load the trained model checkpoint from disk and extract hyperparameters and model state. The checkpoint contains:
- Model state dictionary (weights and biases)
- Hyperparameters for architecture
- Training metrics and metadata

In [2]:
# Load model checkpoint (relative pathname)
model_path = Path(r"../models/251118_model_dff.pth")
checkpoint = torch.load(model_path, map_location=device)

# Print available keys in checkpoint
print(f"Checkpoint keys: {list(checkpoint.keys())}")

# Extract hyperparameters
hyperparams = checkpoint['hyperparams']
print("\n=== Model Hyperparameters ===")
for key, value in hyperparams.items():
    print(f"  {key}: {value}")

C:\Users\dcupolillo\AppData\Local\Temp\ipykernel_11124\1345803451.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(model_path, map_location=device

Checkpoint keys: ['model_state_dict', 'training_dataset', 'train_loss', 'validation_loss', 'train_f1', 'validation_f1', 'validation_precision', 'validation_recall', 'best_thresholds', 'validation_auc_pr', 'valid_epoch_features', 'valid_epoch_labels', 'train_epoch_features', 'train_epoch_labels', 'hyperparams']

=== Model Hyperparameters ===
  learning_rate: 1e-05
  lr_drop_factor: 0.5
  lr_drop_patience: 5
  lambda1: 1e-05
  lambda2: 0.0001
  epochs: 500
  patience: 8
  input_channels: 1
  conv1_channels: 16
  conv2_channels: 32
  conv3_channels: 64
  conv1_kernel: 5
  conv2_kernel: 3
  conv3_kernel: 2
  pool_kernel: 2
  dropout: 0.1
  leaky_relu_negative_slope: 0.05
  batch_size: 16
  random_seed: 1000
  use_dff_only: True


## 2. Initialize Classifier Architecture

Initialize the CalciumEventClassifierDff with hyperparameters from the checkpoint and load the trained weights. Set the model to evaluation mode to disable dropout and batch normalization effects.

In [3]:
classifier = CalciumEventClassifierDff(
    trace_length =              50,  # Adjust if needed based on your data
    input_channels =            hyperparams["input_channels"],
    conv1_channels =            hyperparams["conv1_channels"],
    conv2_channels =            hyperparams["conv2_channels"],
    conv3_channels =            hyperparams["conv3_channels"],
    conv1_kernel =              hyperparams["conv1_kernel"],
    conv2_kernel =              hyperparams["conv2_kernel"],
    conv3_kernel =              hyperparams["conv3_kernel"],
    leaky_relu_negative_slope = hyperparams["leaky_relu_negative_slope"],
    dropout_rate =              hyperparams["dropout"],
    pool_kernel =               hyperparams["pool_kernel"]
).to(device)

# Load model weights and biases
classifier.load_state_dict(checkpoint['model_state_dict'])
classifier.eval()  # Set to evaluation mode

print("✓ Model loaded and ready for inference")

✓ Model loaded and ready for inference


## 3. Display Model Architecture and Parameters

Print the complete model architecture, total/trainable parameters, and detailed parameter statistics including shape, mean, and standard deviation for each layer.

In [4]:
print("\n=== Model Architecture ===")
print(classifier)

# Count parameters
total_params = sum(p.numel() for p in classifier.parameters())
trainable_params = sum(p.numel() for p in classifier.parameters() if p.requires_grad)

print(f"\n=== Parameter Summary ===")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

print(f"\n=== Parameter Statistics ===")
for name, param in classifier.named_parameters():
    if param.requires_grad:
        print(f"  {name:30s} | shape: {str(param.shape):15s} | "
              f"mean: {param.data.mean():8.4f}, std: {param.data.std():8.4f}")


=== Model Architecture ===
CalciumEventClassifierDff(
  (res_block1): ResidualBlock(
    (conv): Conv1d(1, 16, kernel_size=(5,), stride=(1,), padding=same)
    (bn): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act): LeakyReLU(negative_slope=0.05)
    (skip_connection): Conv1d(1, 16, kernel_size=(1,), stride=(1,))
    (skip_bn): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (res_block2): ResidualBlock(
    (conv): Conv1d(16, 32, kernel_size=(3,), stride=(1,), padding=same)
    (bn): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (act): LeakyReLU(negative_slope=0.05)
    (skip_connection): Conv1d(16, 32, kernel_size=(1,), stride=(1,))
    (skip_bn): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (res_block3): ResidualBlock(
    (conv): Conv1d(32, 64, kernel_size=(2,), stride=(1,), padding=same)
    (bn): BatchNorm1d(64, eps=1e-05, m

C:\Users\dcupolillo\AppData\Local\Temp\ipykernel_11124\2166542589.py:16: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\ReduceOps.cpp:1823.)
  f"mean: {param.data.mean():8.4f}, std: {param.data.std():8.4f}")


## 4. Load and Prepare Test Dataset

Load the test dataset from an HDF5 file, create a DffDataset object with baseline normalization, and prepare a DataLoader for batch inference.

In [5]:
# Load test data
test_data_path = Path(r"../datasets/251114_test_dataset.h5")
test_data = fl.load(test_data_path)

# Create test dataset
test_dataset = DffDataset(
    test_data,
    augment=False,  # No augmentation for test
)

# Create test DataLoader
test_loader = cec.load_test_dataset(
    test_dataset,
    batch_size=hyperparams["batch_size"],
    summary=True,
    shuffle=False
)

----------------------------------

 ==== Test Dataset Summary ====
Batch size: 16
Total samples: 456
Test: 456 samples. → 29 batches
   - Label 0: 245 (53.73%) | Label 1: 211 (46.27%)


## 5. Generate Predictions on Test Set

Run the classifier on the test set in inference mode to obtain logits for each sample, then apply sigmoid activation to get prediction probabilities in the range [0, 1].

In [6]:
# Get predictions and labels
labels, logits = cec.get_predictions_and_labels(
    classifier,
    test_loader,
    device
)

# Convert to numpy arrays and apply sigmoid activation
labels = np.array(labels)
predictions = torch.sigmoid(torch.Tensor(logits)).to("cpu").numpy()

print(f"Generated predictions for {len(predictions)} samples")
print(f"Label distribution: {np.bincount(labels.astype(int))}")

Generated predictions for 456 samples
Label distribution: [245 211]


c:\Users\dcupolillo\AppData\Local\anaconda3\envs\2p\lib\site-packages\torch\nn\modules\conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\Convolution.cpp:1037.)
  return F.conv1d(


## 6. Calculate Performance Metrics

Compute the precision-recall curve, find the optimal classification threshold using F1 score, and calculate overall accuracy, precision, recall, and F1 score on the test set.

In [7]:
# Compute precision-recall curve
precision, recall, thresholds = precision_recall_curve(labels, predictions)
precision_ = precision[1:]
recall_ = recall[1:]
f1_scores = 2 * (precision_ * recall_) / (precision_ + recall_ + 1e-8)

# Find optimal threshold using F1 score
best_idx = f1_scores.argmax()
best_thr = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

# Apply optimal threshold to get binary predictions
predicted_classes = (predictions >= best_thr).astype(int)

# Calculate metrics
test_accuracy = accuracy_score(labels, predicted_classes)
test_precision = precision_score(labels, predicted_classes)
test_recall = recall_score(labels, predicted_classes)
test_f1 = f1_score(labels, predicted_classes)

print("\n=== Test Set Performance ===")
print(f"Optimal threshold: {best_thr:.4f}")
print(f"Accuracy:  {test_accuracy:.4f}")
print(f"Precision: {test_precision:.4f}")
print(f"Recall:    {test_recall:.4f}")
print(f"F1 Score:  {test_f1:.4f}")


=== Test Set Performance ===
Optimal threshold: 0.5867
Accuracy:  0.9518
Precision: 0.9315
Recall:    0.9668
F1 Score:  0.9488


## 7. Analyze Confusion Matrix

Calculate and display the confusion matrix with true positives, true negatives, false positives, and false negatives. Also compute error rates and model sensitivity/specificity.

In [8]:
# Compute confusion matrix
cm = confusion_matrix(labels, predicted_classes)
tn, fp, fn, tp = cm.ravel()

print("\n=== Confusion Matrix ===")
print(cm)
print(f"  [[TN={tn:4d}  FP={fp:4d}]")
print(f"   [FN={fn:4d}  TP={tp:4d}]]")

print(f"\n=== Confusion Matrix Breakdown ===")
print(f"  True Negatives (TN):   {tn:4d} - Correctly predicted no event")
print(f"  False Positives (FP):  {fp:4d} - Incorrectly predicted event")
print(f"  False Negatives (FN):  {fn:4d} - Missed actual events")
print(f"  True Positives (TP):   {tp:4d} - Correctly predicted event")

print(f"\n=== Error Rates ===")
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
print(f"  False Positive Rate: {fpr*100:.1f}% ({fp} out of {fp+tn} actual negatives)")
print(f"  False Negative Rate: {fnr*100:.1f}% ({fn} out of {fn+tp} actual positives)")


=== Confusion Matrix ===
[[230  15]
 [  7 204]]
  [[TN= 230  FP=  15]
   [FN=   7  TP= 204]]

=== Confusion Matrix Breakdown ===
  True Negatives (TN):    230 - Correctly predicted no event
  False Positives (FP):    15 - Incorrectly predicted event
  False Negatives (FN):     7 - Missed actual events
  True Positives (TP):    204 - Correctly predicted event

=== Error Rates ===
  False Positive Rate: 6.1% (15 out of 245 actual negatives)
  False Negative Rate: 3.3% (7 out of 211 actual positives)


## 8. Compute Per-Class Metrics

Calculate precision and recall for each class (class 0: no event, class 1: event) based on confusion matrix values.

In [9]:
# Calculate per-class metrics
class0_precision = tn / (tn + fn) if (tn + fn) > 0 else 0
class0_recall = tn / (tn + fp) if (tn + fp) > 0 else 0

class1_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
class1_recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print("\n=== Per-Class Metrics ===")
print(f"Class 0 (No Event):")
print(f"  Precision: {class0_precision:.4f} ({class0_precision*100:.1f}%)")
print(f"  Recall:    {class0_recall:.4f} ({class0_recall*100:.1f}%)")
print(f"\nClass 1 (Event):")
print(f"  Precision: {class1_precision:.4f} ({class1_precision*100:.1f}%)")
print(f"  Recall:    {class1_recall:.4f} ({class1_recall*100:.1f}%)")


=== Per-Class Metrics ===
Class 0 (No Event):
  Precision: 0.9705 (97.0%)
  Recall:    0.9388 (93.9%)

Class 1 (Event):
  Precision: 0.9315 (93.2%)
  Recall:    0.9668 (96.7%)


## 9. Visualize Results

Plot the confusion matrix and probability distribution using custom plotting functions. Also calculate the area under the precision-recall curve (PR-AUC).

In [11]:
# Plot confusion matrix
cec_plot.plot_confusion_matrix(
    labels,
    predicted_classes,
    classes=["No Event", "Event"],
)
plt.tight_layout()
plt.show()

# Plot probability distribution
cec_plot.prob_distribution(
    labels,
    predictions,
)
plt.tight_layout()
plt.show()

# Calculate precision-recall AUC
pr_auc = auc(recall, precision)
print(f"\nArea under PR curve: {pr_auc:.4f}")


Area under PR curve: 0.9809


C:\Users\dcupolillo\AppData\Local\Temp\ipykernel_11124\2238483259.py:15: UserWarning: The figure layout has changed to tight
  plt.tight_layout()


## 10. Perform Confidence and Class-wise Analysis

Analyze prediction confidence by identifying highly confident predictions (far from 0.5 threshold), and evaluate per-class performance to understand where the model excels or struggles.

In [12]:
# Prediction confidence analysis
print("\n=== Prediction Confidence Analysis ===")
confident_predictions = np.abs(predictions - 0.5) > 0.3
print(f"Confident predictions: {confident_predictions.sum()}/{len(predictions)} "
      f"({100 * confident_predictions.mean():.1f}%)")

if confident_predictions.sum() > 0:
    confident_accuracy = accuracy_score(
        labels[confident_predictions],
        predicted_classes[confident_predictions]
    )
    print(f"Accuracy on confident predictions: {confident_accuracy:.4f}")

# Class-wise performance
print("\n=== Class-wise Performance ===")
for class_label in [0, 1]:
    class_mask = labels == class_label
    if class_mask.sum() > 0:
        class_accuracy = accuracy_score(
            labels[class_mask],
            predicted_classes[class_mask]
        )
        class_mean_prob = predictions[class_mask].mean()
        class_name = "No Event" if class_label == 0 else "Event"
        print(f"  {class_name:15s}: accuracy={class_accuracy:.4f}, "
              f"mean_prob={class_mean_prob:.4f}, n={class_mask.sum()}")

print("\n✓ Testing completed!")


=== Prediction Confidence Analysis ===
Confident predictions: 393/456 (86.2%)
Accuracy on confident predictions: 0.9746

=== Class-wise Performance ===
  No Event       : accuracy=0.9388, mean_prob=0.1310, n=245
  Event          : accuracy=0.9668, mean_prob=0.9193, n=211

✓ Testing completed!


## 11. Generate Sorted Heatmap Visualization

Create a heatmap of dFF traces sorted by prediction probability (from high to low) to visualize which patterns the model identifies as events and examine if there are consistent features.

In [13]:
# Create sorted heatmap
sorted_idx = np.argsort(predictions)[::-1]
sorted_dff = test_data["dff"][sorted_idx]
sorted_labels = labels[sorted_idx]

fig, ax = plt.subplots(figsize=(10, 12))
im = ax.imshow(sorted_dff, aspect="auto", cmap="viridis")
ax.axvline(16, color="red", linewidth=2, label="Baseline window end")

# Add colorbar
cbar = plt.colorbar(im, ax=ax, label="dFF")

# Set labels
ax.set_xlabel("Time (samples)", fontsize=12)
ax.set_ylabel("Traces (sorted by prediction probability)", fontsize=12)
ax.set_title("Sorted dFF Heatmap (High → Low Prediction Probability)", fontsize=14, fontweight='bold')
ax.legend()

plt.tight_layout()
plt.show()

print(f"Sorted {len(sorted_dff)} traces by prediction probability")
print(f"Baseline region marked at sample 16 (red line)")

Sorted 456 traces by prediction probability
Baseline region marked at sample 16 (red line)
